In [4]:
import os

# 데이터셋 폴더 안에 뭐가 있는지 확인
for root, dirs, files in os.walk("감정데이터셋"):
    for file in files:
        print(os.path.join(root, file))

감정데이터셋\감정 분류를 위한 대화 음성 데이터셋\4차년도.csv
감정데이터셋\감정 분류를 위한 대화 음성 데이터셋\5차년도.csv
감정데이터셋\감정 분류를 위한 대화 음성 데이터셋\5차년도_2차.csv
감정데이터셋\한국어 감정 정보가 포함된 연속적 대화 데이터셋\한국어_연속적_대화_데이터셋.xlsx


In [6]:
import pandas as pd

base = "감정데이터셋/감정 분류를 위한 대화 음성 데이터셋"

df4 = pd.read_csv(f"{base}/4차년도.csv", encoding="cp949")
df5 = pd.read_csv(f"{base}/5차년도.csv", encoding="cp949")
df5_2 = pd.read_csv(f"{base}/5차년도_2차.csv", encoding="cp949")

for name, df in [("4차년도", df4), ("5차년도", df5), ("5차년도_2차", df5_2)]:
    print(f"=== {name} ===")
    print(f"shape: {df.shape}")
    print(df.columns.tolist())
    print()

=== 4차년도 ===
shape: (14606, 15)
['wav_id', '발화문', '상황', '1번 감정', '1번 감정세기', '2번 감정', '2번 감정세기', '3번 감정', '3번 감정세기', '4번 감정', '4번감정세기', '5번 감정', '5번 감정세기', '나이', '성별']

=== 5차년도 ===
shape: (10011, 15)
['wav_id', '발화문', '상황', '1번 감정', '1번 감정세기', '2번 감정', '2번 감정세기', '3번 감정', '3번 감정세기', '4번 감정', '4번감정세기', '5번 감정', '5번 감정세기', '나이', '성별']

=== 5차년도_2차 ===
shape: (19374, 15)
['wav_id', '발화문', '상황', '1번 감정', '1번 감정세기', '2번 감정', '2번 감정세기', '3번 감정', '3번 감정세기', '4번 감정', '4번감정세기', '5번 감정', '5번 감정세기', '나이', '성별']



In [7]:
# 1번 감정 컬럼 기준으로 전체 감정 종류 확인
all_emotions = pd.concat([
    df4['1번 감정'],
    df5['1번 감정'],
    df5_2['1번 감정']
])

print(all_emotions.value_counts())

1번 감정
Sadness      10591
neutral       6825
Neutral       5133
Angry         4386
happiness     3066
sadness       2824
disgust       2252
Disgust       2125
Fear          1759
surprise      1739
fear          1351
angry         1317
Happiness      443
Surprise       180
Name: count, dtype: int64


In [8]:
# 전체 합치기
df_all = pd.concat([df4, df5, df5_2], ignore_index=True)

# 감정 컬럼 소문자 정규화
emotion_cols = ['1번 감정', '2번 감정', '3번 감정', '4번 감정', '5번 감정']
for col in emotion_cols:
    df_all[col] = df_all[col].str.lower().str.strip()

# 다수결로 최종 감정 결정
from collections import Counter

def majority_emotion(row):
    emotions = [row[col] for col in emotion_cols if pd.notna(row[col]) and row[col] != '']
    if not emotions:
        return None
    return Counter(emotions).most_common(1)[0][0]

df_all['emotion'] = df_all.apply(majority_emotion, axis=1)

# 결과 확인
print(df_all['emotion'].value_counts())
print(f"\n총 샘플 수: {len(df_all)}")

emotion
sadness      16856
neutral       8038
angry         7958
happiness     4061
disgust       2934
fear          2903
surprise      1241
Name: count, dtype: int64

총 샘플 수: 43991


In [10]:
# 파인튜닝용 컬럼만 추출
df_final = df_all[['발화문', 'emotion']].copy()

# 결측값 제거
df_final = df_final.dropna()
df_final = df_final[df_final['발화문'].str.strip() != '']

# 레이블 인코딩
label2id = {
    'happiness': 0,
    'neutral':   1,
    'sadness':   2,
    'angry':     3,
    'disgust':   4,
    'fear':      5,
    'surprise':  6
}
id2label = {v: k for k, v in label2id.items()}

df_final['label'] = df_final['emotion'].map(label2id)

# train/validation/test 분할 (8:1:1)
from sklearn.model_selection import train_test_split

df_train, df_temp = train_test_split(df_final, test_size=0.2, random_state=42, stratify=df_final['label'])
df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=42, stratify=df_temp['label'])

print(f"train : {len(df_train)}")
print(f"val   : {len(df_val)}")
print(f"test  : {len(df_test)}")

# CSV로 저장
df_train.to_csv("감정데이터셋/train.csv", index=False, encoding="utf-8-sig")
df_val.to_csv("감정데이터셋/val.csv", index=False, encoding="utf-8-sig")
df_test.to_csv("감정데이터셋/test.csv", index=False, encoding="utf-8-sig")

print("\n저장 완료!")

train : 35192
val   : 4399
test  : 4400

저장 완료!
